# Stage 3 — Labeling with the OpenAI judge

Drives Stage 3 (`subsec:stage3_labeling`) against the trajectory artifacts of one experiment:

1. queries the judge (`gpt-4o-mini`) through the locked prompt template — resumable, raw responses archived verbatim;
2. shows the label distribution;
3. creates the 60-sample human-labelling CSV and, once you fill it in, runs the Cohen's κ check (`κ ≥ 0.6`, Landis & Koch 1977).

**Requires** `OPENAI_API_KEY` in the environment. Labels are stored separately from trajectories, so relabelling never touches the GPU stages.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from energy_llm.config import load_config

EXPERIMENT = "e1"  # e1 | e2 | e3
cfg = load_config(REPO_ROOT / f"configs/{EXPERIMENT}.yaml")
cfg.output_dir = str(REPO_ROOT / cfg.output_dir)

print(f"experiment:    {cfg.name}")
print(f"trajectories:  {cfg.trajectories_dir}")
print(f"labels:        {cfg.labels_dir}")
print(f"judge:         {cfg.labeling.judge_model}")
assert os.environ.get("OPENAI_API_KEY"), "export OPENAI_API_KEY first"

## Run the judge (resumable)

Already-labelled samples are skipped, so this cell can be re-run after any interruption.

In [ ]:
from energy_llm.labeling import label_samples, make_judge_client

client = make_judge_client(cfg.labeling.api_base)
counts = label_samples(
    cfg.trajectories_dir, cfg.labels_dir, client,
    judge_model=cfg.labeling.judge_model,
    max_retries=cfg.labeling.max_retries,
)
counts

## Label distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from energy_llm.labeling import load_labels

labels = load_labels(cfg.labels_dir)
values = np.array(list(labels.values()))
n_hall = int(values.sum())
print(f"labelled samples:    {len(values)}")
print(f"hallucinations (1):  {n_hall}  ({values.mean():.1%})")
print(f"supported (0):       {len(values) - n_hall}")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(["supported (0)", "hallucination (1)"], [len(values) - n_hall, n_hall],
       color=["#4878d0", "#d65f5f"])
ax.set_ylabel("samples")
ax.set_title(f"{cfg.name}: judge label distribution (n={len(values)})")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
cfg.figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(cfg.figures_dir / "label_distribution.png", dpi=150)
plt.show()

## Human-labelled subset (κ check)

Creates `results/<exp>/human_labels.csv` with 60 random labelled samples (seed 42). Fill the `human_label` column by hand (`0` = supported, `1` = hallucination), then run the κ cell. The questions/answers to grade are shown below for convenience.

In [ ]:
import csv

from energy_llm.io_artifacts import load_json
from energy_llm.labeling import make_human_subset_csv

human_csv = cfg.out / "human_labels.csv"
if human_csv.exists():
    print(f"already exists (not overwritten): {human_csv}")
else:
    make_human_subset_csv(cfg.labels_dir, human_csv,
                          n=cfg.labeling.human_subset_size, seed=cfg.seed)
    print(f"template written: {human_csv}")

# show what needs grading
with open(human_csv) as fh:
    subset_ids = [row["sample_id"] for row in csv.DictReader(fh)]
for sid in subset_ids[:5]:
    meta = load_json(cfg.trajectories_dir / f"{sid}.json")
    print(f"\n[{sid}]\n  Q: {meta['question']}\n  A: {meta['generated_text']!r}\n",
          f" gold: {meta['gold_answers'][:3]}")
print(f"\n... and {len(subset_ids) - 5} more — see the trajectory JSONs.")

In [ ]:
from energy_llm.labeling import judge_human_kappa

try:
    out = judge_human_kappa(cfg.labels_dir, human_csv)
    print(f"Cohen's kappa = {out['kappa']:.3f}   (n = {out['n_subset']})")
    print(f"raw agreement = {out['raw_agreement']:.3f}")
    print("PASSES kappa >= 0.6" if out["passes_threshold"]
          else "BELOW kappa = 0.6 — report results with this limitation stated")
except ValueError as exc:
    print(f"not enough filled human labels yet: {exc}")